# SkyPortal - Fetch candidates for a group

This notebook uses the SkyPortal API to retrieve candidates associated with a given group.

In [3]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

sys.path.insert(0, "utils")
from utils.skyportal_api import SkyPortal

SKYPORTAL_URL = os.getenv("SKYPORTAL_URL")
SKYPORTAL_TOKEN = os.getenv("SKYPORTAL_TOKEN")

if not SKYPORTAL_TOKEN:
    raise ValueError("SKYPORTAL_TOKEN is missing. Set it in the .env file.")

skyPortal = SkyPortal(SKYPORTAL_URL, SKYPORTAL_TOKEN)
print(f"Connected to {SKYPORTAL_URL}")

Connected to https://orcusgate.org/


## 1. List accessible groups

In [4]:
groups = skyPortal.api("GET", "/api/groups")["user_groups"]

groups_df = pd.DataFrame([{"id": g["id"], "name": g["name"]} for g in groups])
groups_df

,id,name
0,7,alecalloch
1,12,LIONS ZTF
2,2,Sitewide Group
3,19,Superphot LSST
4,13,Superphot ZTF


## 2. Select the group and fetch candidates

In [12]:
GROUP_ID = 13
LAST_N_DAYS = 2  # Number of days to look back (1 = last night)
START_DATE = None # Optional: specific start date (overrides LAST_N_DAYS) e.g. "2026-03-04T18:00:00"

In [13]:
from datetime import datetime, timedelta, timezone

start_date = START_DATE or (datetime.now(timezone.utc) - timedelta(days=LAST_N_DAYS)).strftime("%Y-%m-%dT%H:%M:%S")

candidates = skyPortal.fetch_all_pages(
    "/api/candidates",
    payload={"groupIDs": str(GROUP_ID), "startDate": start_date},
    item_key="candidates",
)
print(f"{len(candidates)} candidates passed since {start_date} UTC")

10 candidates passed since 2026-03-02T22:08:09 UTC


## 3. Display results

In [14]:
df = pd.DataFrame([
    {
        "id": c["id"],
        "ra": c.get("ra"),
        "dec": c.get("dec"),
        "redshift": c.get("redshift"),
        "last_detected_at": c.get("last_detected_at"),
        "classifications": ", ".join(
            cl["classification"] for cl in c.get("classifications", [])
        ),
    }
    for c in candidates
])

print(f"{len(df)} candidates for group {GROUP_ID}")
df.head(20)

10 candidates for group 13


,id,ra,dec,redshift,last_detected_at,classifications
0,ZTF26aahyyny,120.848618,8.290481,None,2026-03-03T07:17:32.000637,
1,ZTF26aahpthq,132.348036,18.525050,None,2026-03-03T07:05:47.002570,
2,ZTF26aahfrag,138.980906,10.135896,None,2026-03-03T07:03:43.001271,
3,ZTF26aahaesb,132.356133,29.510539,None,2026-03-03T06:58:05.998070,
4,ZTF26aahajej,123.776604,17.822087,None,2026-03-03T05:51:16.001273,
5,ZTF26aahauhi,114.896892,27.636411,None,2026-03-03T05:58:51.000978,
6,ZTF26aabpbkj,130.820612,3.689420,None,2026-03-03T05:52:38.003524,
7,ZTF26aabjpnx,144.567295,26.688913,None,2026-03-03T04:08:22.004144,
8,ZTF26aaclbkw,72.821795,0.427953,None,2026-03-03T03:34:09.001922,
9,ZTF25acgklik,72.001587,-16.661684,None,2026-03-03T03:04:43.003213,
